# Slice R&D Benchmark (PFF→in-memory L1/L2)

Measures PFF-slice→L1→features→inference vs full materialization, across slice sizes and decimation factors.

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
OBS_DIR = Path("/path/to/obs.pffd")  # replace
DP = "dp_img16.bpp_2.module_1"  # replace with actual product name

results: list[BenchResult] = []

## §1 Full materialization baseline (convert + calibrate)

In [ ]:
from panoseti_analysis.adapters.calibrate import run_calibrate
from panoseti_analysis.adapters.convert import run_convert

OUT_BASE = Path("/tmp/slice_rnd_bench")
CONVERT_OUT = OUT_BASE / "convert"
CALIBRATE_OUT = OUT_BASE / "calibrate"

with stage_timer("full_pipeline_baseline", bytes_in=0) as r:
    records_convert = run_convert(OBS_DIR, CONVERT_OUT, checksum=False)

    # Find the L0 store for this product and module
    l0_stores = list(CONVERT_OUT.glob("**/*.L0.zarr"))
    if l0_stores:
        l0_store = l0_stores[0]
        records_calibrate = run_calibrate(l0_store, CALIBRATE_OUT)
        out_bytes = sum(f.stat().st_size for f in CALIBRATE_OUT.rglob("*") if f.is_file())
        r.bytes_out = out_bytes

results.append(r)
print(summarize(results))

## §2 In-memory slice (slice_to_l1)

In [ ]:
# from panoseti_analysis.adapters.slice_driver import slice_to_l1  # Phase 1b
# from panoseti_analysis.io.pff import open_pff_product              # Phase 1b
# seq = open_pff_product(OBS_DIR, DP, module=1)
# with stage_timer("slice_to_l1_10s", bytes_in=0) as r:
#     ds_l1 = slice_to_l1(seq, time_range=(t_start_ns, t_start_ns + 10*int(1e9)), decimate=10)
# results.append(r)
# TODO: uncomment after Phase 1b lands
print("Phase 1b not yet implemented — uncomment cells above after slice_driver.py is added")

## §3 Summary

In [ ]:
print(summarize(results))